# Task A / A1：100题单步DA探针（独立Colab T4 Notebook）

本Notebook已按用户最终确认采用冻结Stage 1规格：A0/B0各独立取20–79；每步u/v各独立取1–9。
比较旧状态，同时更新：A>B时(A-u,B+v)，否则（含相等）(A+u,B-v)。允许负数，不取模、不裁剪。
**最终答案为有序整数对(A,B)，不进行数值编码。** 本次仅d=1、100题、DA，不运行CoT、Task B或正式深度实验。

完整内嵌冻结Stage 1与v3定义单元，源代码与原文件逐字节相同。直接使用v3精确判分与DA泄漏诊断，
不重新定义解析规则；若旧评分器抛出异常，外围记录原文、异常和失败分母并暂停。

文档中的0.80–0.95与0.955–0.986两个窗口原样并列；本次只测p1_DA，不测p1_CoT，
不把二者视为同一个概率、不自动进入A2、不根据模型结果调整本批数据。

**Colab执行顺序：** 配置 → 环境/T4检查 → 冻结定义与评分接入 → CPU测试 → Drive/100题 → 模型 → 首题 → 剩余题 → 汇总。
首题计入100次预算；断点只继续未发送题目。OOM、截断、评分异常或结果未知请求会停止并保存记录，不自动重发。
首次权重下载约8GB（约7.5GiB），另需依赖和缓存空间。T4通常约15GiB显存，vLLM预算90%。
实际模型revision、vLLM/PyTorch/CUDA/GPU会落盘；CPU测试不能代替T4运行验证。


## 1. 已确认的规格与冻结来源

In [8]:
# 已明确批准：冻结Stage 1转移与分布，最终答案为有序整数对。
TRANSITION_SPEC = 'frozen_stage1'
FROZEN_SOURCES = {'dual_register_spec.ipynb': {'notebook_sha256': '1209871b6498b527735b0a4ac74f20708d87da645b063e88f0f112ef84535028', 'source_sha256': '27ffa745256bde3b318cf6646c5a45cbddbc32ed70e65959963e64b4a6e7a071'}, 'free_text_scoring_v3.ipynb': {'notebook_sha256': '64d94370186d4c10fd413dc0c813c4872e82960ed471931584097b8468387f86', 'source_sha256': 'b2c3e6e21f342a2a6e3fe0aa155ccca1d0ae4977ebcc03bd011a8c11c7342aab'}}

## 2. Colab环境与T4检查、缺失依赖安装
CPU侧验证跳过本单元。沿用历史Notebook的缺失依赖安装方式；已有vLLM不升级。安装失败或提示重启时先停止处理，不自动降配。

In [9]:
# 仅在用户自己的 Colab T4 执行；CPU测试不会执行本单元。
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('google.colab') is None:
    raise RuntimeError('本推理入口只允许在 Google Colab T4 运行。CPU验证请只执行标记的测试单元。')
check=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],
                     check=True,capture_output=True,text=True)
print(check.stdout)
if 'T4' not in check.stdout:
    raise RuntimeError('未检测到T4；不自动切换模型或修改配置。')
# 沿用旧calibration的安装方式；若已有vLLM，保留其版本，不自动升级。
missing=[name for name in ('vllm','datasets') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
    print('依赖已安装；如Colab提示重启，请重启后再从本单元开始。')


Tesla T4, 15360 MiB, 580.82.07



## 3. 冻结Stage 1实现
下方源代码与已有Notebook的定义单元逐字节一致；不执行历史测试或历史数据生成单元。

In [10]:
# 所有实验实现保存在 notebook；此源片段由构建工具写入 notebook 代码单元。
import json
import random
import re
from collections import Counter

SPEC_VERSION = 'dual-register-cpu-v1'


def integer(value, name):
    # 排除 bool 和隐式浮点转换，避免悄悄改变输入。
    if type(value) is not int:
        raise ValueError(f'{name} must be an integer')
    return value


def pair(value, name='state'):
    if not isinstance(value, (tuple, list)) or len(value) != 2:
        raise ValueError(f'{name} must contain two integers')
    return tuple(integer(x, name) for x in value)


def parameters(operations):
    result = [pair(op, 'operation') for op in operations]
    if not result or any(not (1 <= u <= 9 and 1 <= v <= 9) for u, v in result):
        raise ValueError('nonempty operations with u,v in 1..9 required')
    return result


def simulate(initial, operations):
    """参考解：状态包含起点，分支用 GT / LE 表示，始终读取旧状态。"""
    a, b = pair(initial)
    ops = parameters(operations)
    states, branches = [[a, b]], []
    for u, v in ops:
        old_a, old_b = a, b
        gt = old_a > old_b
        a = old_a - u if gt else old_a + u
        b = old_b + v if gt else old_b - v
        states.append([a, b])
        branches.append('GT' if gt else 'LE')
    return {'states': states, 'branches': branches, 'answer': [a, b]}


def simulate_ds(initial, operations):
    """独立 D/S 路径：不调用 A/B 模拟器或转移函数。"""
    a, b = pair(initial)
    difference, total = a - b, a + b
    states, branches = [[a, b]], []
    for u, v in parameters(operations):
        sign = 1 if difference > 0 else -1
        difference = difference - sign * (u + v)
        total = total + sign * (v - u)
        assert (total + difference) % 2 == 0
        states.append([(total + difference) // 2, (total - difference) // 2])
        branches.append('GT' if sign == 1 else 'LE')
    return {'states': states, 'branches': branches, 'answer': states[-1].copy()}


def workload(depth, width=1):
    integer(depth, 'depth'); integer(width, 'width')
    if depth < 1 or width < 1:
        raise ValueError('positive depth and width required')
    return {'W_update': depth * width, 'W_core': 3 * depth * width}


def simulate_parallel(chains):
    """计算独立不等于模型内部并行；不共享状态，不做答案聚合。"""
    if not chains:
        raise ValueError('at least one chain required')
    depths = [len(c['operations']) for c in chains]
    if len(set(depths)) != 1:
        raise ValueError('this parallel reference requires equal chain depths')
    trajectories = [simulate(c['initial'], c['operations']) for c in chains]
    return {'trajectories': trajectories,
            'answers': [t['answer'].copy() for t in trajectories],
            **workload(depths[0], len(chains))}


def render_question(initial, operations):
    a, b = initial
    rows = '\n'.join(f'{t}. u={u}, v={v}' for t, (u, v) in enumerate(operations, 1))
    return (f'Initially A={a} and B={b}.\n'
            'Each numbered row is one complete transition. Compare the OLD A and B. '
            'If A > B, set A to old A minus u and B to old B plus v. '
            'Otherwise (including equality), set A to old A plus u and B to old B minus v. '
            'Both assignments use the old state. Negative integers are allowed; '
            'do not apply modulo or clipping.\n' + rows + '\n'
            'What are the final values of A and B, in that order?')


def make_question(seed, depth, qid=None):
    integer(seed, 'seed'); integer(depth, 'depth')
    if not 1 <= depth <= 20:
        raise ValueError('validation specification supports depth 1..20')
    rng = random.Random(seed)
    initial = [rng.randint(20, 79), rng.randint(20, 79)]
    operations = [[rng.randint(1, 9), rng.randint(1, 9)] for _ in range(depth)]
    reference = simulate(initial, operations)
    return {'spec_version': SPEC_VERSION, 'purpose': 'cpu_validation_only',
            'qid': qid or f'd{depth}-s{seed}', 'seed': seed, 'd': depth,
            'initial': initial, 'operations': operations, **workload(depth),
            'q': render_question(initial, operations), 'ans': reference['answer'].copy(),
            'reference': reference}


def make_validation_dataset(seed=20261107, depths=(1, 5, 10, 15, 20), per_depth=20):
    # 独立子种子可重建每条题；不按轨迹、答案或模型表现拒绝任何样本。
    integer(per_depth, 'per_depth')
    if per_depth < 1:
        raise ValueError('positive per_depth required')
    rng = random.Random(seed)
    return [make_question(rng.getrandbits(63), d, f'cpu-d{d}-i{i:03d}')
            for d in depths for i in range(per_depth)]


# 解析器仅接受完整最终作答，不在推理全文中搜索碰巧正确的数字。
NUMBER = r'[+-]?[0-9]+'
PAIR_PATTERN = rf'\(\s*({NUMBER})\s*,\s*({NUMBER})\s*\)'
LABEL_PATTERN = rf'A\s*=\s*({NUMBER})\s*,\s*B\s*=\s*({NUMBER})'


def parse_answer(text):
    """支持裸整数对、A/B 标注及显式最终答案行；歧义一律不猜。"""
    if not isinstance(text, str) or not text.strip():
        return {'parse_ok': False, 'answer': None, 'reason': 'empty_or_nontext'}
    lines = [line.strip() for line in text.replace('\u2212', '-').splitlines() if line.strip()]
    def parse_body(body):
        for pattern in (PAIR_PATTERN, LABEL_PATTERN):
            match = re.fullmatch(pattern, body, flags=re.I)
            if match:
                return [int(match[1]), int(match[2])]
        return None
    if len(lines) == 1:
        value = parse_body(lines[0])
        if value is not None:
            return {'parse_ok': True, 'answer': value, 'reason': 'bare_answer'}
    finals = [(i, re.fullmatch(r'(?:Final answer|最终答案)\s*[:：]\s*(.*)', line, re.I))
              for i, line in enumerate(lines)]
    finals = [(i, m) for i, m in finals if m]
    if len(finals) != 1:
        return {'parse_ok': False, 'answer': None, 'reason': 'missing_or_multiple_final_answers'}
    index, match = finals[0]
    if index != len(lines) - 1:
        return {'parse_ok': False, 'answer': None, 'reason': 'text_after_final_answer'}
    # 未标注的独立答案行与显式最终答案同时出现时，保守拒绝。
    if any(parse_body(line) is not None for line in lines[:index]):
        return {'parse_ok': False, 'answer': None, 'reason': 'competing_answer'}
    value = parse_body(match[1])
    return {'parse_ok': value is not None, 'answer': value,
            'reason': 'explicit_final' if value is not None else 'invalid_final_pair'}


def score_answer(text, gold):
    expected = list(pair(gold, 'gold'))
    parsed = parse_answer(text)
    return {**parsed, 'final_answer_ok': parsed['parse_ok'] and parsed['answer'] == expected}


def trajectory_metrics(reference, predicted_states=None, predicted_branches=None):
    """结构化预测按深度提供；不从自然语言臆测状态或分支。"""
    d = len(reference['branches'])
    result = {'state_accuracy': None, 'first_error_depth': None, 'branch_accuracy': None,
              'recovery_rate': None, 'recovery_count': 0, 'recovery_opportunities': 0,
              'state_coverage': 0.0, 'branch_coverage': 0.0}
    if predicted_states is not None:
        if len(predicted_states) != d:
            raise ValueError('predicted_states must have d entries; use None for missing steps')
        states = [None if s is None else list(pair(s)) for s in predicted_states]
        correct = [s == reference['states'][t + 1] for t, s in enumerate(states)]
        result['state_accuracy'] = sum(correct) / d
        result['state_coverage'] = sum(s is not None for s in states) / d
        result['first_error_depth'] = next((i + 1 for i, ok in enumerate(correct) if not ok), None)
        eligible = [i for i in range(d - 1)
                    if states[i] is not None and states[i + 1] is not None and not correct[i]]
        recovered = sum(correct[i + 1] for i in eligible)
        result.update(recovery_count=recovered, recovery_opportunities=len(eligible),
                      recovery_rate=recovered / len(eligible) if eligible else None)
    if predicted_branches is not None:
        if len(predicted_branches) != d or any(b not in ('GT', 'LE', None) for b in predicted_branches):
            raise ValueError('branches must have d entries from GT, LE, None')
        result['branch_accuracy'] = sum(a == b for a, b in zip(predicted_branches, reference['branches'])) / d
        result['branch_coverage'] = sum(b is not None for b in predicted_branches) / d
    return result


def evaluate_response(text, reference, predicted_states=None, predicted_branches=None):
    return {**score_answer(text, reference['answer']),
            **trajectory_metrics(reference, predicted_states, predicted_branches)}


def trajectory_diagnostics(initial, operations):
    ref = simulate(initial, operations)
    states, branches = ref['states'], ref['branches']
    d = len(branches)
    switches = sum(a != b for a, b in zip(branches, branches[1:]))
    run = longest = 1
    for before, after in zip(branches, branches[1:]):
        run = run + 1 if before == after else 1
        longest = max(longest, run)
    cancellations = sum(states[t] == states[t - 2] for t in range(2, d + 1))
    # 合流测量使用固定初值邻居，共享完全相同的参数序列，不改变数据生成。
    # 统计“上一步完整状态不同、下一步完整状态相同”的事件及机会数。
    a, b = pair(initial)
    events = opportunities = 0
    for neighbor in ((a + 2, b - 2), (a - 2, b + 2), (a + 4, b - 2), (a - 4, b + 2)):
        other = simulate(neighbor, operations)['states']
        for t in range(1, d + 1):
            if states[t - 1] != other[t - 1]:
                opportunities += 1
                events += states[t] == other[t]
    return {'branch_switch_count': switches, 'branch_switch_opportunities': d - 1,
            'branch_switching_rate': switches / (d - 1) if d > 1 else None,
            'longest_same_branch_run': longest,
            'state_convergence_count': events, 'state_convergence_opportunities': opportunities,
            'state_convergence_frequency': events / opportunities if opportunities else None,
            'cancellation_count': cancellations, 'cancellation_opportunities': max(0, d - 1),
            'cancellation_frequency': cancellations / (d - 1) if d > 1 else None}


## 4. 冻结Stage 1.5 v3评分器
直接复用`score_free_text`解析两个最终整数并精确比较有序对。无标量编码适配层。

In [11]:
import re

SCORING_VERSION = 'task-a-free-text-v3'
_INT = r'[+-]?[0-9]{1,64}'
_PAIR = rf'\(\s*({_INT})\s*,\s*({_INT})\s*\)'
_LABELS = rf'A\s*=\s*({_INT})\s*[,;]\s*B\s*=\s*({_INT})'
_MARKER = re.compile(r'^(?:Final answer|Final values|Answer|最终答案|最终值)(?:\s*[:：]\s*(.*))?$', re.I)
_DECLARATION = re.compile(r'^(?:The final values are|The final answer is)\s+(.+)$', re.I)


def _normalize_line(line):
    # 只在解析副本中规范化；原始响应由评分记录原样保留。
    line = line.strip().replace('\u2212', '-')
    line = re.sub(r'^#{1,6}\s+', '', line)
    if line.startswith('**') and line.endswith('**') and line.count('**') == 2 and len(line) > 4:
        line = line[2:-2].strip()
    line = re.sub(r'\*\*([^*]+)\*\*', r'\1', line)
    if line.startswith('`') and line.endswith('`') and not line.startswith('```'):
        line = line[1:-1].strip()
    return line


def _pair_body(body):
    body = _normalize_line(body)
    if body.endswith('.'):
        body = body[:-1].rstrip()
    for pattern in (_PAIR, _LABELS):
        match = re.fullmatch(pattern, body, re.I)
        if match:
            return [int(match[1]), int(match[2])]
    return None


def parse_final_answer(raw_response, finish_reason=None):
    """v3：相同完整最终声明去重，冲突或畸形声明不回退；与 gold 无关。"""
    if not isinstance(raw_response, str):
        raise TypeError('raw_response must be a string; use empty text for empty completions')
    if finish_reason not in (None, 'stop', 'length', 'abort', 'error'):
        raise ValueError('unsupported finish_reason; map provider metadata explicitly')
    items = [(i, _normalize_line(line)) for i,line in enumerate(raw_response.splitlines()) if line.strip()]
    def fail(reason, ambiguous=False):
        return {'answer_parse_ok':False, 'parsed_answer':None, 'parse_reason':reason,
                'answer_line_indices':[], 'ambiguity':ambiguous, 'final_declaration_count':0}
    def success(value, indices, reason, count):
        return {'answer_parse_ok':True, 'parsed_answer':value, 'parse_reason':reason,
                'answer_line_indices':indices, 'ambiguity':False, 'final_declaration_count':count}
    if not items:
        return fail('empty')
    if any('```' in line for _,line in items):
        return fail('unsupported_code_fence')
    finals=[]
    for pos,(_,line) in enumerate(items):
        match=_MARKER.fullmatch(line) or _DECLARATION.fullmatch(line)
        if match:
            finals.append((pos,match[1]))
        elif re.search(r'\b(?:final answer|final values|answer is)\b|最终答案|最终值',line,re.I):
            return fail('unsupported_final_declaration')
    if finals:
        values, indices, consumed = [], [], set()
        for k,(pos,body) in enumerate(finals):
            boundary=finals[k+1][0] if k+1<len(finals) else len(items)
            span=[pos]
            if body:
                value=_pair_body(body)
            else:
                following=items[pos+1:boundary]
                value=_pair_body(following[0][1]) if following else None
                if value is not None:
                    span.append(pos+1)
                elif len(following)>=2:
                    value=_pair_body(following[0][1]+', '+following[1][1])
                    if value is not None:span.extend([pos+1,pos+2])
            if value is None:
                # 显式多个候选即歧义；缺项/半截答案为语法失败，均不得回退。
                alternatives=len(re.findall(_PAIR,body))>1 or bool(re.search(r'\bor\b|或者|或是',body,re.I))
                return fail('ambiguous_final_alternatives' if alternatives else 'malformed_final_answer', alternatives)
            values.append(value)
            # 与 items 和返回的 indices 一致，统一保存原始响应行号。
            consumed.update(items[p][0] for p in span)
            indices.extend(items[p][0] for p in span)
        if len({tuple(v) for v in values})!=1:
            return fail('conflicting_final_declarations',True)
        if any(_pair_body(line) is not None for raw_index,line in items
               if raw_index not in consumed):
            return fail('unlabelled_competing_answer',True)
        # 声明之后的额外材料可能是在撤回或改写答案。只允许 length 时的无数字、
        # 无不确定/否定措辞的说明尾段；其余保守拒绝，防止截断造成误判。
        last=max(consumed)
        tail=[line for raw_index,line in items if raw_index>last]
        if tail:
            text=' '.join(tail)
            safe_tail=(finish_reason=='length' and not re.search(
                r'[0-9=()]|\b(?:not|wrong|incorrect|actually|instead|maybe|perhaps|wait|or)\b|错|不|或',text,re.I))
            if not safe_tail:return fail('text_after_final_answer')
        return success(values[0],indices,'explicit_final' if len(values)==1 else 'identical_final_declarations',len(values))
    if len(items)==1:
        value=_pair_body(items[0][1])
    elif len(items)==2:
        value=_pair_body(', '.join(line for _,line in items))
    else:
        value=None
    if value is not None:
        if finish_reason in ('length','abort','error'):
            return fail('unfinished_unmarked_answer')
        return success(value,[i for i,_ in items],'answer_only',0)
    bare_candidates=[_pair_body(line) for _,line in items if _pair_body(line) is not None]
    if len(bare_candidates)>1 or (len(items)==1 and len(re.findall(_PAIR,items[0][1]))>1):
        return fail('multiple_unmarked_candidates',True)
    return fail('no_unambiguous_final_answer')


def _da_diagnostics(raw_response, parsed, mode):
    if mode != 'DA':
        return {'da_extra_text': None, 'da_arith': None, 'da_trace': None,
                'da_trace_evidence_count': None}
    answer_lines = set(parsed['answer_line_indices'])
    residual = [line.strip() for i, line in enumerate(raw_response.splitlines())
                if i not in answer_lines and line.strip()]
    # 即使最终声明冲突，也不能把声明本身误作推理泄漏证据。
    # 这里排除的是诊断候选行，不改变 parse_ok 或主评分。
    source = [(i, _normalize_line(line)) for i,line in enumerate(raw_response.splitlines()) if line.strip()]
    candidate_lines = set(answer_lines)
    for pos,(i,line) in enumerate(source):
        declaration = _MARKER.fullmatch(line) or _DECLARATION.fullmatch(line)
        if declaration:
            candidate_lines.add(i)
            if not declaration[1]:
                following = source[pos+1:pos+3]
                if following and _pair_body(following[0][1]) is not None:
                    candidate_lines.add(following[0][0])
                elif len(following)==2 and _pair_body(following[0][1]+', '+following[1][1]) is not None:
                    candidate_lines.update(j for j,_ in following)
    reasoning = [line.strip() for i,line in enumerate(raw_response.splitlines())
                 if i not in candidate_lines and line.strip()]
    # 启发式诊断，不作为最终答案准确率的过滤条件。
    def arithmetic(line):
        numeric = re.search(r'[0-9]', line)
        return bool(numeric and (
            re.search(r'[0-9]\s*[+\-−*/×÷]\s*[+\-−]?[0-9]', line)
            or re.search(r'\b(?:add|subtract|plus|minus|multiply|divide)\b|加上|减去|乘以|除以', line, re.I)
            or re.search(r'\b[AB]\s*[+\-]=\s*[0-9]', line, re.I)))
    def state_claim(line):
        return bool(re.search(r'\b[AB]\s*=\s*[+\-−]?[0-9]', line, re.I)
                    or re.search(_PAIR, line.replace('−','-'))
                    or re.search(r'(?:\bstep\s*[0-9]+|第\s*[0-9]+\s*步)', line, re.I))
    evidence_lines = sum(arithmetic(line) or state_claim(line) for line in reasoning)
    step_labels = set(re.findall(r'\bstep\s*([0-9]+)|第\s*([0-9]+)\s*步', '\n'.join(reasoning), re.I))
    evidence = max(evidence_lines, len(step_labels))
    return {'da_extra_text': bool(residual), 'da_arith': any(arithmetic(line) for line in reasoning),
            'da_trace': evidence >= 2, 'da_trace_evidence_count': evidence}


def score_free_text(raw_response, gold, mode, finish_reason=None):
    if mode not in ('DA', 'CoT'):
        raise ValueError('mode must be DA or CoT')
    if not isinstance(gold, (tuple,list)) or len(gold) != 2 or any(type(x) is not int for x in gold):
        raise ValueError('gold must be an ordered integer pair')
    parsed = parse_final_answer(raw_response, finish_reason)
    answer_ok = parsed['answer_parse_ok'] and parsed['parsed_answer'] == list(gold)
    diagnostics = _da_diagnostics(raw_response, parsed, mode)
    return {'scoring_version': SCORING_VERSION, 'raw_response': raw_response,
            'mode': mode, 'finish_reason': finish_reason, **parsed,
            'answer_ok': answer_ok, 'final_answer_ok': answer_ok,
            'a_ok': parsed['answer_parse_ok'] and parsed['parsed_answer'][0] == gold[0],
            'b_ok': parsed['answer_parse_ok'] and parsed['parsed_answer'][1] == gold[1],
            'da_leakage': (diagnostics['da_arith'] or diagnostics['da_trace']) if mode == 'DA' else None,
            'parse_failure': not parsed['answer_parse_ok'],
            'truncation': None if finish_reason is None else finish_reason == 'length',
            'truncated': None if finish_reason is None else finish_reason == 'length',
            'runtime_failure': finish_reason in ('abort', 'error'),
            **diagnostics}


def summarize_scores(records):
    """主分母包含全部记录；诊断同时给出分子/分母，不静默排除失败。"""
    if any(r['scoring_version']!=SCORING_VERSION for r in records):
        raise ValueError('do not mix scoring versions')
    n=len(records)
    parsed=sum(r['answer_parse_ok'] for r in records)
    correct=sum(r['answer_ok'] for r in records)
    rate=lambda a,b: a/b if b else None
    known=[r for r in records if r['truncation'] is not None]
    da=[r for r in records if r['mode']=='DA']
    out={'n_total':n,'n_parsed':parsed,'n_correct':correct,
         'primary_answer_accuracy':rate(correct,n),'main_accuracy':rate(correct,n),
         'parse_success_rate':rate(parsed,n),'parse_failure_count':n-parsed,
         'parse_failure_rate':rate(n-parsed,n),'conditional_answer_accuracy':rate(correct,parsed),
         'ambiguity_count':sum(r['ambiguity'] for r in records),
         'ambiguity_rate':rate(sum(r['ambiguity'] for r in records),n),
         'truncation_known_n':len(known),'truncation_unknown_n':n-len(known),
         'truncation_count':sum(r['truncation'] for r in known),
         'truncation_rate':rate(sum(r['truncation'] for r in known),len(known)),
         'runtime_failure_count':sum(r['runtime_failure'] for r in records),
         'da_n':len(da)}
    for name in ('da_extra_text','da_arith','da_trace','da_leakage'):
        numerator=sum(r[name] for r in da)
        out[name+'_count']=numerator
        out[name+'_rate']=rate(numerator,len(da))
    for component in ('a','b'):
        numerator=sum(r[component+'_ok'] for r in records)
        out[component+'_correct_count']=numerator
        out[component+'_accuracy']=rate(numerator,n)
        out[component+'_conditional_accuracy']=rate(numerator,parsed)
    return out


## 5. 探针配置、落盘与请求记账
固定temperature=0.6、top_p=0.95、DA max_tokens=512；不显式传top_k或thinking开关。只发送题干，参考答案和种子不进入prompt。

In [12]:
import hashlib
import math
import os
import time
from datetime import datetime,timezone
from pathlib import Path

PROBE_VERSION='task-a-da-pair-probe-v1'
MODEL_ID='Qwen/Qwen3-4B-Instruct-2507'
DATA_SEED=2026110801
MODEL_SEED=2026110803
MODEL_KW=dict(dtype='half',max_model_len=8192,gpu_memory_utilization=0.90,max_num_seqs=32)
SAMPLE_KW=dict(temperature=0.6,top_p=0.95,max_tokens=512)
DA_PROMPT='Give only the final ordered integer pair (A, B), without reasoning or intermediate steps.'


def sha(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()


def timestamp():
    return datetime.now(timezone.utc).isoformat()


def read_rows(path):
    if not Path(path).exists():
        return []
    # 损坏/半行直接停止，不能忽略后重发请求。
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def append_row(path,row):
    with Path(path).open('a') as stream:
        stream.write(json.dumps(row,ensure_ascii=False)+'\n')
        stream.flush()
        os.fsync(stream.fileno())


def save_once(path,text):
    path=Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        if path.read_text()!=text:
            raise RuntimeError(f'已有文件不一致，停止而非覆盖: {path}')
    else:
        with path.open('x') as stream:
            stream.write(text)
            stream.flush()
            os.fsync(stream.fileno())


def messages_for(q):
    return [{'role':'system','content':DA_PROMPT},{'role':'user','content':q['question']}]


def request_status(out_dir):
    out_dir=Path(out_dir)
    attempts=read_rows(out_dir/'request_attempts_v1.jsonl')
    results=read_rows(out_dir/'probe_results_v1.jsonl')
    ids=[r['question_id'] for r in attempts]
    result_ids=[r['question_id'] for r in results]
    if len(ids)!=len(set(ids)) or len(result_ids)!=len(set(result_ids)):
        raise RuntimeError('重复发送/结果，需人工审查')
    if not set(result_ids).issubset(ids):
        raise RuntimeError('有结果缺少发送记录')
    return attempts,results


def summary_rows(out_dir):
    # 断线后已发送但未收到的请求仍进入主分母，并单列状态未知。
    attempts,results=request_status(out_dir)
    by_id={r['question_id']:r for r in results}
    reconciled=[]
    for attempt in attempts:
        if attempt['question_id'] in by_id:
            reconciled.append(by_id[attempt['question_id']])
        else:
            row=score_probe_response('',attempt['gold_answer'],None)
            row.update(attempt,raw_response_available=False,runtime_failure=True,da_leakage=None,
                       generation_error='interrupted_outcome_unknown',elapsed_seconds=None,
                       completion_tokens=None)
            reconciled.append(row)
    return reconciled


def run_requests(out_dir,questions,manifest,runtime,generate_one,prompt_length,limit=None):
    out_dir=Path(out_dir)
    if runtime['protocol_sha256']!=manifest['protocol_sha256']:
        raise RuntimeError('运行环境与数据协议不一致')
    attempts,results=request_status(out_dir)
    planned={q['question_id']:q for q in questions}
    for attempt in attempts:
        q=planned.get(attempt['question_id'])
        if q is None or attempt['protocol_sha256']!=manifest['protocol_sha256'] or attempt['seed']!=q['model_seed']:
            raise RuntimeError('发送记录与冻结计划不一致')
    done={r['question_id'] for r in results}
    if {a['question_id'] for a in attempts}!=done:
        raise RuntimeError('存在已发送但结果未知请求；已计入分母，禁止自动重发或继续')
    if any(r['runtime_failure'] or r['truncation'] for r in results) or (out_dir/'HALTED.json').exists():
        raise RuntimeError('存在运行失败/截断；保留全部记录，停止等待审查')
    pending=[q for q in questions if q['question_id'] not in done]
    if limit is not None:
        pending=pending[:limit]
    if not pending:
        print('没有未发送请求；不重复推理。')
        return results
    lock=out_dir/'run.lock'
    # 双击或另一个标签页不得同时发送；异常断线留下的锁须人工检查。
    with lock.open('x') as stream:
        stream.write(timestamp())
    try:
        # 在取得锁后重新检查，避免两个调用使用同一旧快照。
        if request_status(out_dir)!=(attempts,results):
            raise RuntimeError('请求状态已变化，请重新检查')
        for q in pending:
            messages=messages_for(q)
            n_prompt=prompt_length(messages)
            if n_prompt+SAMPLE_KW['max_tokens']>MODEL_KW['max_model_len']:
                raise RuntimeError('输入超过上下文预算；未发送，不修改配置')
            attempt=dict(question_id=q['question_id'],depth=1,mode='DA',seed=q['model_seed'],
                         question_seed=q['question_seed'],initial=q['initial'],operations=q['operations'],
                         gold_pair=q['gold_pair'],gold_answer=q['gold_answer'],messages=messages,
                         protocol_sha256=manifest['protocol_sha256'],model_id=MODEL_ID,
                         model_revision=runtime['model_revision'],environment_sha256=runtime['environment_sha256'],
                         dispatched_at=timestamp(),prompt_tokens=n_prompt)
            append_row(out_dir/'request_attempts_v1.jsonl',attempt)
            raw,finish,n_completion,error='', 'error',None,None
            start=time.perf_counter()
            try:
                response=generate_one(messages,q)
                raw=response['raw_response']
                finish=response['finish_reason']
                n_prompt=response['prompt_tokens']
                n_completion=response['completion_tokens']
                if not isinstance(raw,str) or finish not in ('stop','length','abort','error'):
                    raise ValueError('不支持的生成返回值')
                if type(n_prompt) is not int or type(n_completion) is not int:
                    raise ValueError('缺少token计数')
            except BaseException as exc:
                error=f'{type(exc).__name__}: {exc}'
                finish='error'
                raw=raw if isinstance(raw,str) else repr(raw)
            elapsed=time.perf_counter()-start
            # 原始生成先落盘；评分异常也不会丢失模型回答。
            generation=dict(**attempt,raw_response=raw,finish_reason=finish,
                            completion_tokens=n_completion,elapsed_seconds=elapsed,
                            generation_error=error,received_at=timestamp())
            generation['prompt_tokens']=n_prompt
            append_row(out_dir/'raw_generations_v1.jsonl',generation)
            score=score_probe_response(raw,q['gold_answer'],finish)
            scoring_error=score['scoring_error']
            row={**generation,**score,'scoring_error':scoring_error,'raw_response_available':error is None or bool(raw)}
            if not row['raw_response_available'] or scoring_error:
                row['da_leakage']=None
            append_row(out_dir/'probe_results_v1.jsonl',row)
            results.append(row)
            if error or finish!='stop' or scoring_error:
                reason=error or scoring_error or f'finish_reason={finish}'
                save_once(out_dir/'HALTED.json',json.dumps(dict(reason=reason,question_id=q['question_id']),ensure_ascii=False)+'\n')
                raise RuntimeError(f'已保存并停止：{reason}。不得自动重试或调整配置。')
            print(f'{len(results)}/100 {q["question_id"]}: answer={row["answer_ok"]}, '
                  f'parse={row["answer_parse_ok"]}, tokens={n_completion}')
        return results
    finally:
        lock.unlink()


## 6. v3评分接入与统计
外围只保存异常，不改变提取策略。解析失败、截断和运行失败均保留在主分母；A、B分量准确率也按全部已发送请求统计。

In [13]:
def score_probe_response(raw_response,gold,finish_reason):
    # 直接使用冻结v3评分；这里只记录异常，不修改答案提取规则。
    try:
        return {**score_free_text(raw_response,gold,'DA',finish_reason),'scoring_error':None}
    except Exception as exc:
        # 某些不支持的畸形声明可能触发旧解析器异常；保留原文和全部分母并暂停。
        fallback=score_free_text('',gold,'DA',finish_reason)
        return {**fallback,'raw_response':raw_response,'parse_reason':'scoring_exception',
                'da_extra_text':None,'da_arith':None,'da_trace':None,'da_leakage':None,
                'da_trace_evidence_count':None,'scoring_error':f'{type(exc).__name__}: {exc}'}


def wilson95(correct,total):
    if total==0:
        return None
    z=1.959963984540054
    rate=correct/total
    denominator=1+z*z/total
    centre=(rate+z*z/(2*total))/denominator
    half=z*math.sqrt(rate*(1-rate)/total+z*z/(4*total*total))/denominator
    return [max(0.,centre-half),min(1.,centre+half)]


def summarize_probe(records,planned=100):
    n=len(records)
    parsed=sum(r['answer_parse_ok'] for r in records)
    correct=sum(r['answer_ok'] for r in records)
    known=[r for r in records if r['truncation'] is not None]
    rate=lambda a,b: a/b if b else None
    return dict(planned=planned,attempted=n,not_attempted=planned-n,complete=n==planned,
                correct=correct,parsed=parsed,main_accuracy=rate(correct,n),main_wilson95=wilson95(correct,n),
                parse_success_rate=rate(parsed,n),parse_wilson95=wilson95(parsed,n),
                conditional_accuracy=rate(correct,parsed),conditional_wilson95=wilson95(correct,parsed),
                parse_failure_rate=rate(n-parsed,n),
                ambiguity_rate=rate(sum(r['ambiguity'] for r in records),n),
                truncation_rate=rate(sum(r['truncation'] for r in known),len(known)),
                truncation_known_n=len(known),truncation_unknown_n=n-len(known),
                da_leakage_rate=rate(sum(r['da_leakage'] is True for r in records),n),
                da_leakage_unknown_n=sum(r['da_leakage'] is None for r in records),
                runtime_failure_count=sum(r['runtime_failure'] for r in records),
                scoring_error_count=sum(bool(r.get('scoring_error')) for r in records),
                a_accuracy=rate(sum(r['a_ok'] for r in records),n),
                b_accuracy=rate(sum(r['b_ok'] for r in records),n),
                a_wilson95=wilson95(sum(r['a_ok'] for r in records),n),
                b_wilson95=wilson95(sum(r['b_ok'] for r in records),n),
                decision='等待人工明确目标窗口；本Notebook不自动进入A2',
                da_window_080_095=(0.80<=correct/n<=0.95) if n==planned else None,
                reference_window_0955_0986=(0.955<=correct/n<=0.986) if n==planned else None,
                warning='两个窗口分别列示，不合并；本次未测p1_CoT，不能用p1_DA证明CoT深度趋势')


## 7. d=1生成器与JSONL回读
直接调用冻结生成器；固定63位子种子，无编码或额外算术步骤。发现重复题则报告，不重采样。

In [14]:
def make_probe_questions():
    data_rng=random.Random(DATA_SEED)
    model_rng=random.Random(MODEL_SEED)
    questions=[]
    for i in range(100):
        seed=data_rng.getrandbits(63)
        # 原样调用冻结生成器，完整保留题干和有序对答案，无编码和重采样。
        original=make_question(seed,1,f'pair-da-v1-i{i:03d}')
        questions.append(dict(question_id=original['qid'],depth=1,question_seed=seed,
                              model_seed=model_rng.randrange(2**31),initial=original['initial'],
                              operations=original['operations'],gold_pair=original['ans'],
                              gold_answer=original['ans'].copy(),question=original['q'],
                              W_update=original['W_update'],W_core=original['W_core'],
                              spec_version=original['spec_version'],purpose='A1_DA_model_probe'))
    # 碰到重复题则停止报告，不按模型表现或轨迹筛选、补抽。
    assert len({sha([q['initial'],q['operations']]) for q in questions})==100
    return questions


def prepare_probe_inputs(out_dir):
    from datasets import load_dataset
    out_dir=Path(out_dir)
    questions=make_probe_questions()
    text=''.join(json.dumps(q,ensure_ascii=False)+'\n' for q in questions)
    save_once(out_dir/'probe_dataset_v1.jsonl',text)
    loaded=load_dataset('json',data_files=str(out_dir/'probe_dataset_v1.jsonl'),split='train')
    assert list(loaded)==questions
    for q in questions:
        assert simulate(q['initial'],q['operations'])['answer']==q['gold_answer']==q['gold_pair']
    manifest=dict(probe_version=PROBE_VERSION,transition_spec=TRANSITION_SPEC,answer_format='ordered_integer_pair',
                  model_id=MODEL_ID,question_count=100,depths=[1],modes=['DA'],planned_requests=100,
                  data_seed=DATA_SEED,model_seed=MODEL_SEED,dataset_sha256=hashlib.sha256(text.encode()).hexdigest(),
                  frozen_sources=FROZEN_SOURCES,scoring_version=SCORING_VERSION,
                  da_prompt=DA_PROMPT,model_config=MODEL_KW,sampling_config=SAMPLE_KW,
                  target_conflict={'DA_d1':[0.80,0.95],'p1_reference':[0.955,0.986]},auto_advance=False)
    manifest['protocol_sha256']=sha(manifest)
    save_once(out_dir/'probe_manifest_v1.json',json.dumps(manifest,ensure_ascii=False,indent=2)+'\n')
    return questions,manifest


## 8. CPU接口验证
人工模型返回值只存在临时目录，不进入Drive结果；100题生成、固定种子重建和datasets回读均在CPU执行。模型调用数为0。

In [15]:
import contextlib
import io
import tempfile
import unittest


class PairScoringTests(unittest.TestCase):
    def test_clear_pair_formats_and_negatives(self):
        for raw,gold in [('(0,-12)',[0,-12]),('(+21,19)',[21,19]),
                         ('**Final answer:**\n**A=-12**\n**B=0**',[-12,0]),
                         ('### Final answer\n\n(-12,0)',[-12,0]),
                         ('Final answer: `(21,19)`',[21,19])]:
            with self.subTest(raw=raw):
                r=score_probe_response(raw,gold,'stop')
                self.assertTrue(r['answer_parse_ok']);self.assertTrue(r['answer_ok'])
                self.assertFalse(r['ambiguity']);self.assertEqual(r['raw_response'],raw)
                self.assertIsNone(r['scoring_error'])

    def test_intermediate_never_replaces_final(self):
        for raw in ('Step 1: (21,19)','Step 1: (21,19)\nFinal answer:','Final answer: (21,'):
            r=score_probe_response(raw,[21,19],'stop')
            self.assertFalse(r['answer_parse_ok']);self.assertFalse(r['answer_ok'])
        r=score_probe_response('Step 1: (21,19)\nFinal answer: (19,21)',[21,19],'stop')
        self.assertTrue(r['answer_parse_ok']);self.assertFalse(r['answer_ok'])
        self.assertFalse(r['a_ok']);self.assertFalse(r['b_ok'])

    def test_identical_and_conflicting_final_pairs(self):
        r=score_probe_response('Final answer: (21,19)\nFinal answer: (21,19)',[21,19],'stop')
        self.assertTrue(r['answer_ok']);self.assertFalse(r['ambiguity'])
        self.assertEqual(r['parse_reason'],'identical_final_declarations')
        r=score_probe_response('Final answer: (21,19)\nFinal answer: (19,21)',[21,19],'stop')
        self.assertFalse(r['answer_parse_ok']);self.assertTrue(r['ambiguity'])
        self.assertEqual(r['parse_reason'],'conflicting_final_declarations')

    def test_blank_lines_conflicts_and_retractions(self):
        for raw in ('\nFinal answer: (21,19)\n(19,21)',
                    '\nFinal answer:\nA=21\nB=19\n\n(19,21)'):
            r=score_probe_response(raw,[21,19],'stop')
            self.assertFalse(r['answer_parse_ok']);self.assertTrue(r['ambiguity'])
            self.assertEqual(r['parse_reason'],'unlabelled_competing_answer')
        r=score_probe_response('\nFinal answer: (21,19)\n\nWait, wrong',[21,19],'length')
        self.assertFalse(r['answer_parse_ok']);self.assertFalse(r['ambiguity'])
        self.assertEqual(r['parse_reason'],'text_after_final_answer')

    def test_truncation_preserves_clear_answer(self):
        for raw in ('Final answer: (21,19)','\n\nFinal answer:\n(21,19)\n\nI checked the'):
            r=score_probe_response(raw,[21,19],'length')
            self.assertTrue(r['answer_ok']);self.assertTrue(r['truncation'])
        for raw in ('(21,19)','Final answer:','Final answer: (21,'):
            r=score_probe_response(raw,[21,19],'length')
            self.assertFalse(r['answer_parse_ok']);self.assertTrue(r['truncation'])

    def test_empty_incomplete_and_scalar_are_rejected(self):
        for raw in ('','text','2119','21','(21,)','(21,19,0)','Final answer: 2119'):
            self.assertFalse(score_probe_response(raw,[21,19],'stop')['answer_parse_ok'])

    def test_v3_exception_recorded_without_changing_parser(self):
        r=score_probe_response('Final answer',[21,19],'stop')
        self.assertFalse(r['answer_ok']);self.assertFalse(r['answer_parse_ok'])
        self.assertEqual(r['raw_response'],'Final answer')
        self.assertEqual(r['parse_reason'],'scoring_exception')
        self.assertIsNotNone(r['scoring_error']);self.assertIsNone(r['da_leakage'])

    def test_leakage_does_not_filter_accuracy(self):
        r=score_probe_response('20+1=21\nFinal answer: (21,19)',[21,19],'stop')
        self.assertTrue(r['answer_ok']);self.assertTrue(r['da_leakage'])
        self.assertEqual(r['scoring_version'],'task-a-free-text-v3')

    def test_main_parse_conditional_and_component_denominators(self):
        rows=[score_probe_response(s,[21,19],f) for s,f in
              [('(21,19)','stop'),('(21,18)','stop'),('','error'),('','length')]]
        stats=summarize_probe(rows)
        self.assertEqual(stats['main_accuracy'],1/4)
        self.assertEqual(stats['parse_success_rate'],2/4)
        self.assertEqual(stats['conditional_accuracy'],1/2)
        self.assertEqual(stats['a_accuracy'],2/4);self.assertEqual(stats['b_accuracy'],1/4)
        self.assertEqual(stats['runtime_failure_count'],1)
        self.assertEqual(stats['truncation_rate'],1/4)
        self.assertFalse(stats['complete']);self.assertIsNone(stats['da_window_080_095'])

    def test_wilson_known_and_empty(self):
        low,high=wilson95(96,100)
        self.assertAlmostEqual(low,0.9016292856,places=8)
        self.assertAlmostEqual(high,0.9843366960,places=8)
        self.assertIsNone(wilson95(0,0))
        self.assertAlmostEqual(wilson95(0,100)[0],0)
        self.assertAlmostEqual(wilson95(100,100)[1],1)


class PersistenceTests(unittest.TestCase):
    def setUp(self):
        self.temp=tempfile.TemporaryDirectory()
        self.addCleanup(self.temp.cleanup)
        self.out=Path(self.temp.name)
        # 人工接口桩，不是真实模型输出，也不属于探针题目。
        self.questions=[dict(question_id=f'fake-{i}',initial=[20,20],operations=[[1,1]],
                             gold_pair=[21,19],gold_answer=[21,19],question='test input',
                             question_seed=i,model_seed=i) for i in range(2)]
        self.manifest={'protocol_sha256':'cpu-test-only'}
        self.runtime=dict(protocol_sha256='cpu-test-only',model_revision='fake',environment_sha256='fake')
        self.calls=[]

    def generate(self,messages,q):
        self.calls.append(q['question_id'])
        return dict(raw_response='(21,19)',finish_reason='stop',prompt_tokens=10,completion_tokens=2)

    def run_probe(self,generate=None,limit=None):
        with contextlib.redirect_stdout(io.StringIO()):
            return run_requests(self.out,self.questions,self.manifest,self.runtime,
                                generate or self.generate,lambda messages:10,limit)

    def test_resume_never_repeats_completed_requests(self):
        self.run_probe(limit=1)
        self.run_probe()
        self.run_probe()
        self.assertEqual(self.calls,['fake-0','fake-1'])
        attempts,rows=request_status(self.out)
        self.assertEqual(len(attempts),2);self.assertEqual(len(rows),2)
        self.assertEqual(rows[0]['prompt_tokens'],10)
        self.assertEqual(rows[0]['completion_tokens'],2)
        self.assertGreaterEqual(rows[0]['elapsed_seconds'],0)
        self.assertEqual(len(read_rows(self.out/'raw_generations_v1.jsonl')),2)

    def test_runtime_error_retained_and_stops(self):
        def fail(messages,q):
            self.calls.append(q['question_id'])
            raise RuntimeError('synthetic OOM')
        with self.assertRaises(RuntimeError):self.run_probe(fail)
        self.assertEqual(len(self.calls),1)
        rows=summary_rows(self.out)
        self.assertEqual(len(rows),1);self.assertFalse(rows[0]['answer_ok'])
        self.assertTrue(rows[0]['runtime_failure'])
        self.assertIsNone(rows[0]['completion_tokens'])
        with self.assertRaises(RuntimeError):self.run_probe()
        self.assertEqual(len(self.calls),1)

    def test_complete_truncated_answer_scores_but_stops(self):
        def truncated(messages,q):
            return dict(raw_response='Final answer: (21,19)',finish_reason='length',prompt_tokens=10,completion_tokens=512)
        with self.assertRaises(RuntimeError):self.run_probe(truncated)
        rows=summary_rows(self.out)
        self.assertTrue(rows[0]['answer_ok']);self.assertTrue(rows[0]['truncation'])
        self.assertEqual(len(rows),1)

    def test_interrupted_attempt_in_denominator_not_retried(self):
        q=self.questions[0]
        append_row(self.out/'request_attempts_v1.jsonl',dict(question_id=q['question_id'],seed=q['model_seed'],
                   gold_answer=q['gold_answer'],protocol_sha256=self.manifest['protocol_sha256']))
        rows=summary_rows(self.out)
        self.assertEqual(len(rows),1);self.assertTrue(rows[0]['runtime_failure'])
        self.assertFalse(rows[0]['answer_ok']);self.assertIsNone(rows[0]['truncation'])
        self.assertEqual(summarize_probe(rows)['main_accuracy'],0)
        with self.assertRaises(RuntimeError):self.run_probe()
        self.assertEqual(self.calls,[])

    def test_context_failure_is_not_dispatched(self):
        with self.assertRaises(RuntimeError):
            run_requests(self.out,self.questions,self.manifest,self.runtime,self.generate,lambda messages:8192)
        self.assertEqual(request_status(self.out),([],[]));self.assertEqual(self.calls,[])

    def test_existing_data_never_overwritten(self):
        path=self.out/'fixture.json'
        save_once(path,'original');save_once(path,'original')
        with self.assertRaises(RuntimeError):save_once(path,'replacement')
        self.assertEqual(path.read_text(),'original')

    def test_duplicate_journal_and_lock_rejected(self):
        record=dict(question_id='fake-0')
        append_row(self.out/'request_attempts_v1.jsonl',record)
        append_row(self.out/'request_attempts_v1.jsonl',record)
        with self.assertRaises(RuntimeError):request_status(self.out)
        (self.out/'request_attempts_v1.jsonl').unlink()
        save_once(self.out/'run.lock','existing session')
        with self.assertRaises(FileExistsError):self.run_probe()
        self.assertEqual(self.calls,[])


class ProbeInputTests(unittest.TestCase):
    def test_dataset_seed_rebuild_exact_question_and_gold(self):
        questions=make_probe_questions()
        self.assertEqual(questions,make_probe_questions())
        self.assertEqual(len(questions),100)
        self.assertEqual({q['depth'] for q in questions},{1})
        self.assertEqual(len({q['question_id'] for q in questions}),100)
        for q in questions:
            ref=make_question(q['question_seed'],1,q['question_id'])
            self.assertEqual(q['initial'],ref['initial'])
            self.assertEqual(q['operations'],ref['operations'])
            self.assertEqual(q['gold_answer'],ref['ans'])
            self.assertEqual(q['question'],ref['q'])
            self.assertEqual(q['gold_answer'],simulate_ds(q['initial'],q['operations'])['answer'])
            self.assertEqual(q['W_update'],1);self.assertEqual(q['W_core'],3)
            self.assertTrue(all(20<=x<=79 for x in q['initial']))
            self.assertEqual(len(q['operations']),1)
            self.assertTrue(all(1<=x<=9 for x in q['operations'][0]))

    def test_dataset_jsonl_and_datasets_readback(self):
        with tempfile.TemporaryDirectory() as tmp:
            questions,manifest=prepare_probe_inputs(tmp)
            self.assertEqual(read_rows(Path(tmp)/'probe_dataset_v1.jsonl'),questions)
            self.assertEqual(manifest['planned_requests'],100)
            self.assertEqual(manifest['modes'],['DA'])
            self.assertEqual(manifest['answer_format'],'ordered_integer_pair')
            self.assertFalse(manifest['auto_advance'])
            self.assertEqual(manifest['scoring_version'],'task-a-free-text-v3')
            self.assertEqual(prepare_probe_inputs(tmp),(questions,manifest))

    def test_prompt_contains_only_question_and_da_instruction(self):
        for q in make_probe_questions():
            messages=messages_for(q)
            self.assertEqual(messages,[{'role':'system','content':DA_PROMPT},
                                       {'role':'user','content':q['question']}])
            self.assertIn('(A, B)',DA_PROMPT)
            self.assertNotIn('encoded',DA_PROMPT)
            self.assertNotIn('A * 100',q['question'])
            self.assertEqual(set(messages[1]),{'role','content'})


suite=unittest.TestSuite([unittest.defaultTestLoader.loadTestsFromTestCase(cls)
                         for cls in (PairScoringTests,PersistenceTests,ProbeInputTests)])
log=io.StringIO()
result=unittest.TextTestRunner(stream=log,verbosity=2).run(suite)
CPU_TEST_RESULTS=dict(tests_run=result.testsRun,failures=len(result.failures),errors=len(result.errors),
                      skipped=len(result.skipped),model_requests=0)
print(log.getvalue())
print(json.dumps(CPU_TEST_RESULTS,ensure_ascii=False,indent=2))
assert result.wasSuccessful(), 'CPU测试未通过，不得运行模型'


Generating train split: 0 examples [00:00, ? examples/s]

test_blank_lines_conflicts_and_retractions (__main__.PairScoringTests.test_blank_lines_conflicts_and_retractions) ... ok
test_clear_pair_formats_and_negatives (__main__.PairScoringTests.test_clear_pair_formats_and_negatives) ... ok
test_empty_incomplete_and_scalar_are_rejected (__main__.PairScoringTests.test_empty_incomplete_and_scalar_are_rejected) ... ok
test_identical_and_conflicting_final_pairs (__main__.PairScoringTests.test_identical_and_conflicting_final_pairs) ... ok
test_intermediate_never_replaces_final (__main__.PairScoringTests.test_intermediate_never_replaces_final) ... ok
test_leakage_does_not_filter_accuracy (__main__.PairScoringTests.test_leakage_does_not_filter_accuracy) ... ok
test_main_parse_conditional_and_component_denominators (__main__.PairScoringTests.test_main_parse_conditional_and_component_denominators) ... ok
test_truncation_preserves_clear_answer (__main__.PairScoringTests.test_truncation_preserves_clear_answer) ... ok
test_v3_exception_recorded_without_cha

## 9. 挂载Drive、冻结100题与计划
结果目录固定；重连沿用同一路径。既有输入和manifest不一致时停止，不覆盖。运行结束后可从Drive下载所有JSONL及汇总。

In [16]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR=Path('/content/drive/MyDrive/CISC7404/Experiments/task_a/task_a_probe_v1')
OUT_DIR.mkdir(parents=True,exist_ok=True)
QUESTIONS,MANIFEST=prepare_probe_inputs(OUT_DIR)
save_once(OUT_DIR/'cpu_test_results_v1.json',json.dumps(CPU_TEST_RESULTS,indent=2)+'\n')
print('已冻结',len(QUESTIONS),'道题；输出目录：',OUT_DIR)
print('已有请求/结果数：',tuple(len(x) for x in request_status(OUT_DIR)))


Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

已冻结 100 道题；输出目录： /content/drive/MyDrive/CISC7404/Experiments/task_a/task_a_probe_v1
已有请求/结果数： (0, 0)


## 10. 加载Qwen3-4B-Instruct-2507
若已有100次请求，跳过本单元和生成单元，直接汇总。不修改任何加载参数来绕过失败；失败信息保存到Drive。实际模型SHA在首次加载前解析并固定，续跑复用同一SHA。

In [17]:
# 先完成题目/计划落盘，再加载GPU；沿用已核实T4参数，不做自动降配。
os.environ['VLLM_USE_FLASHINFER_SAMPLER']='0'
import torch
import importlib.metadata
from huggingface_hub import HfApi

if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('需要NVIDIA T4，不允许CPU推理或替代模型。')
attempts,existing=request_status(OUT_DIR)
if len(attempts)>=100:
    raise RuntimeError('100次请求已登记；跳过加载和生成，直接执行汇总单元。')
revision_path=OUT_DIR/'model_revision_v1.json'
if revision_path.exists():
    resolved=json.loads(revision_path.read_text())
    if resolved['model_id']!=MODEL_ID:
        raise RuntimeError('已有模型标识不符')
else:
    # 先解析实际commit SHA，再用同一SHA加载模型与tokenizer。
    resolved=dict(model_id=MODEL_ID,revision=HfApi().model_info(MODEL_ID).sha)
    if not re.fullmatch(r'[0-9a-f]{40}',resolved['revision'] or ''):
        raise RuntimeError('无法锁定模型revision')
    save_once(revision_path,json.dumps(resolved,indent=2)+'\n')

from vllm import LLM, SamplingParams
versions={name:importlib.metadata.version(name) for name in ('vllm','torch','transformers','datasets','huggingface-hub')}
identity=dict(model_id=MODEL_ID,model_revision=resolved['revision'],versions=versions,
              python=sys.version,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),
              gpu_memory_bytes=torch.cuda.get_device_properties(0).total_memory,
              gpu_capability=list(torch.cuda.get_device_capability(0)),
              nvidia_smi=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],
                                        check=True,capture_output=True,text=True).stdout.strip(),
              flashinfer_sampler=os.environ['VLLM_USE_FLASHINFER_SAMPLER'],
              llm_config=MODEL_KW,sampling_config=SAMPLE_KW,
              protocol_sha256=MANIFEST['protocol_sha256'])
RUNTIME={**identity,'environment_sha256':sha(identity)}
# 重连时环境不同则停止，避免悄悄混合不同软件/硬件结果。
save_once(OUT_DIR/'runtime_v1.json',json.dumps(RUNTIME,ensure_ascii=False,indent=2)+'\n')
freeze=subprocess.run([sys.executable,'-m','pip','freeze'],check=True,capture_output=True,text=True).stdout
save_once(OUT_DIR/'requirements_actual_v1.txt',freeze)
try:
    llm=LLM(model=MODEL_ID,revision=resolved['revision'],tokenizer_revision=resolved['revision'],**MODEL_KW)
except BaseException as error:
    append_row(OUT_DIR/'load_failures_v1.jsonl',dict(time=timestamp(),error=f'{type(error).__name__}: {error}',**RUNTIME))
    raise

tokenizer=llm.get_tokenizer()
save_once(OUT_DIR/'chat_template_v1.json',json.dumps(tokenizer.chat_template,ensure_ascii=False)+'\n')


def count_prompt(messages):
    return len(tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True))


def generate_one(messages,q):
    params=SamplingParams(**SAMPLE_KW,seed=q['model_seed'])
    output=llm.chat(messages=messages,sampling_params=params,use_tqdm=False)
    if len(output)!=1 or len(output[0].outputs)!=1:
        raise RuntimeError('期望每次请求只返回一个completion')
    item=output[0]
    completion=item.outputs[0]
    return dict(raw_response=completion.text,finish_reason=completion.finish_reason,
                prompt_tokens=len(item.prompt_token_ids),completion_tokens=len(completion.token_ids))

print(json.dumps(RUNTIME,ensure_ascii=False,indent=2))


W1107 13:40:35.318000 447 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1107 13:40:35.320000 447 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1107 13:40:35.323000 447 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


(MainProcess pid=447) INFO 11-07 13:40:48 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'disable_log_stats': True, 'revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'tokenizer_revision': 'cdbee75f17c01a7cc42f958dc650907174af0554', 'model': 'Qwen/Qwen3-4B-Instruct-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

(MainProcess pid=447) INFO 11-07 13:41:09 [model.py:744] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=447) WARNING 11-07 13:41:09 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=447) INFO 11-07 13:41:09 [model.py:2170] Using max model len 8192
(MainProcess pid=447) INFO 11-07 13:41:09 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=447) INFO 11-07 13:41:10 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=447) WARNING 11-07 13:41:15 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=447) INFO 11-07 13:45:13 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.
{
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "versions": {
    "vllm": "0.33.0",
    "torch": "2.14.0",
    "transformers": "5.20.2",
    "datasets": "4.8.5",
    "huggingface-hub": "1.33.0"
  },
  "python": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "cuda": "13.0",
  "gpu": "Tesla T4",
  "gpu_memory_bytes": 15637086208,
  "gpu_capability": [
    7,
    5
  ],
  "nvidia_smi": "Tesla T4, 15360 MiB, 580.82.07",
  "flashinfer_sampler"

## 11. 首题接口检查（计入100次总预算）
验证真实输出、token计数和评分字段；答案算错或解析失败也保留，不能替换样本。出现length、OOM或其他异常时自动停止。

In [18]:
rows=run_requests(OUT_DIR,QUESTIONS,MANIFEST,RUNTIME,generate_one,count_prompt,limit=1)
if rows:
    print(json.dumps(rows[-1],ensure_ascii=False,indent=2))


1/100 pair-da-v1-i000: answer=False, parse=True, tokens=9
{
  "question_id": "pair-da-v1-i000",
  "depth": 1,
  "mode": "DA",
  "seed": 1369091199,
  "question_seed": 4129003157026930875,
  "initial": [
    41,
    51
  ],
  "operations": [
    [
      1,
      8
    ]
  ],
  "gold_pair": [
    42,
    43
  ],
  "gold_answer": [
    42,
    43
  ],
  "messages": [
    {
      "role": "system",
      "content": "Give only the final ordered integer pair (A, B), without reasoning or intermediate steps."
    },
    {
      "role": "user",
      "content": "Initially A=41 and B=51.\nEach numbered row is one complete transition. Compare the OLD A and B. If A > B, set A to old A minus u and B to old B plus v. Otherwise (including equality), set A to old A plus u and B to old B minus v. Both assignments use the old state. Negative integers are allowed; do not apply modulo or clipping.\n1. u=1, v=8\nWhat are the final values of A and B, in that order?"
    }
  ],
  "protocol_sha256": "7b06b823d

## 12. 完成剩余请求
首题正常完成后执行。重跑单元只处理未发送题目，不重复首题或已完成记录。正常耗时取决于T4运行时；预算上限100×512=51200个输出token，不额外发送预热问答。

In [19]:
rows=run_requests(OUT_DIR,QUESTIONS,MANIFEST,RUNTIME,generate_one,count_prompt)
print('已保存模型结果：',len(rows))


2/100 pair-da-v1-i001: answer=True, parse=True, tokens=9
3/100 pair-da-v1-i002: answer=False, parse=True, tokens=9
4/100 pair-da-v1-i003: answer=False, parse=True, tokens=9
5/100 pair-da-v1-i004: answer=False, parse=False, tokens=152
6/100 pair-da-v1-i005: answer=False, parse=True, tokens=9
7/100 pair-da-v1-i006: answer=False, parse=False, tokens=412
8/100 pair-da-v1-i007: answer=False, parse=True, tokens=9
9/100 pair-da-v1-i008: answer=True, parse=True, tokens=9
10/100 pair-da-v1-i009: answer=False, parse=True, tokens=9
11/100 pair-da-v1-i010: answer=False, parse=True, tokens=9
12/100 pair-da-v1-i011: answer=False, parse=True, tokens=9
13/100 pair-da-v1-i012: answer=False, parse=True, tokens=9
14/100 pair-da-v1-i013: answer=False, parse=True, tokens=9
15/100 pair-da-v1-i014: answer=False, parse=True, tokens=9
16/100 pair-da-v1-i015: answer=False, parse=True, tokens=9
17/100 pair-da-v1-i016: answer=False, parse=True, tokens=9
18/100 pair-da-v1-i017: answer=False, parse=False, tokens=23

RuntimeError: 已保存并停止：finish_reason=length。不得自动重试或调整配置。

## 13. 汇总与失败样例（出现停止后也可手动运行）

- 主准确率 = 正确数 / 全部已发送请求数；结果未知和运行失败计未正确，不能删除。
- Parse success = 成功解析数 / 全部已发送数；Conditional = 正确数 / 成功解析数。
- 未发送题目单列，未完成100题时不发布完整p1结论。
- 截断率分母为finish_reason已知的记录数，另报未知数；泄漏率同时报告诊断未知数。
- 解析失败不自动证明算术失败；可解析错误也可能含编码错误。失败原文保留供人工检查。
- 不运行趋势检验或CoT实验。本阶段只有d=1，不能据此证明深度梯度。


In [ ]:
if 'OUT_DIR' not in globals():
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR=Path('/content/drive/MyDrive/CISC7404/Experiments/task_a/task_a_probe_v1')
all_records=summary_rows(OUT_DIR)
stats=summarize_probe(all_records)
examples=[r for r in all_records if not r['answer_ok'] or r['truncation'] or r['da_leakage']]
report=dict(probe_version=PROBE_VERSION,model_inference='not_run' if not all_records else 'attempted',
            metrics=stats,parse_reasons=dict(Counter(r['parse_reason'] for r in all_records)),
            failure_examples=examples[:5],notes=[
                '只测p1_DA，不推断p1_CoT。',
                '解析失败、截断和运行失败没有从主分母删除。',
                '两个验收窗口尚未统一；需要人工审查后再进入A2。',
                '本批参数与样本不因结果变动；后续候选须另起版本。'])
(OUT_DIR/'probe_summary_v1.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(report,ensure_ascii=False,indent=2))


## 14. 审查与后续边界

带回：`probe_results_v1.jsonl`、`raw_generations_v1.jsonl`、`request_attempts_v1.jsonl`、
`probe_dataset_v1.jsonl`、`probe_manifest_v1.json`、`runtime_v1.json`和`probe_summary_v1.json`；若异常附`HALTED.json`或加载失败日志。
实际环境还记录在`requirements_actual_v1.txt`；环境不同则暂停，不能悄悄混跑。

断线后遗留的`run.lock`或已发送无结果请求不能直接删除重跑。先检查Drive记录并报告未知请求；
未知请求进入失败分母，但不冒充已观察到的模型回答。正常断点只继续未发送题目。

v3仍是有限语法的保守解析器：部分LaTeX、JSON、代码围栏和任意散文不受支持。
对于不带冒号且没有答案体的`Final answer`，冻结v3可能抛出异常；本Notebook只捕获、记录并暂停，
不改历史评分器、不猜测答案。保留原文供审查；DA诊断不是可靠的内部推理测量。

转移规则、分布和有序对答案已经确认。尚待实验结果返回后人工决定A1目标窗口及是否进入A2。
仅A2通过人工审查后才生成正式串行及工作量配平的并行数据集。当前无CoT或GPU实测结论。
